# 06 - Training and Internal-Validation Tuning

This notebook trains all classical models. Sampling is seed-controlled, scalers fit on training only, and confidence thresholds are selected only on duplicate-group-separated internal validation to maximize decided-case macro-F1 at at least 80% coverage. The official labelled validation split is not opened for tuning.

In [1]:
import json,sys
from pathlib import Path
cwd=Path.cwd().resolve(); REPO=next((p for p in [cwd,*cwd.parents] if (p/'implementation'/'src').exists()),None)
if REPO is None: raise RuntimeError('Start Jupyter inside the repository.')
sys.path.insert(0,str(REPO/'implementation'/'src'))
from ipcv_attire.dataset_policy import load_policy
from ipcv_attire.training import train_pipeline
TRAINING_PROFILE='full'  # change to full on the 32 GB desktop
FORCE_RETRAIN=False
DATA=REPO/'implementation'/'data'; BUNDLE=REPO/'implementation'/'models'/f'classical-attire-{TRAINING_PROFILE}'


## Laptop and desktop profiles

`smoke` runs the real end-to-end Fashionpedia path on a fixed small subset. `full` uses all relevant samples, float32 disk-backed feature caches, and one hard-negative-mining pass. The RTX 3080 is intentionally unused: every implemented method is classical and CPU-based.

In [2]:
manifest_path=BUNDLE/'manifest.json'
if FORCE_RETRAIN and manifest_path.exists(): raise RuntimeError('Remove the exact ignored bundle directory manually before forcing retraining.')
if not manifest_path.exists(): train_pipeline(manifest_dir=DATA/'interim'/'manifests',fashionpedia_root=DATA/'raw'/'fashionpedia',policy=load_policy(DATA/'dataset-policy.json'),bundle_dir=BUNDLE,profile_name=TRAINING_PROFILE)
metadata=json.loads(manifest_path.read_text(encoding='utf-8')); print(json.dumps(metadata,indent=2))


{
  "pipeline_version": "classical-attire-v1",
  "policy_version": 3,
  "policy_sha256": "866ee9a49ffd0705814f214c8674770334e4049c49b059396c0d4d103986c68d",
  "profile": "full",
  "feature_config": {
    "width": 64,
    "height": 128,
    "hog_orientations": 9,
    "hog_pixels_per_cell": [
      8,
      8
    ],
    "hog_cells_per_block": [
      2,
      2
    ],
    "lbp_points": 8,
    "lbp_radius": 1,
    "hsv_bins": [
      8,
      4,
      4
    ]
  },
  "trained_targets": [
    "allowed_sleeve",
    "casual_or_tight_bottom",
    "casual_round_neck_top",
    "collared_top",
    "damaged_bottom",
    "formal_bottom_candidate",
    "leisurewear",
    "revealing_top"
  ],
  "detector_components": [
    "bottom_body",
    "footwear",
    "headwear",
    "upper_body"
  ],
  "environment": {
    "numpy": "2.5.1",
    "opencv": "4.13.0",
    "pillow": "12.3.0",
    "scikit-image": "0.26.0",
    "scikit-learn": "1.9.0"
  },
  "bundle_sha256": "69e17521316bd46cdb8e85c13f277d436140eb1c2

## Full desktop command

```powershell
python implementation/src/train_classical_pipeline.py --profile full --bundle-dir implementation/models/classical-attire-full
```

The ignored model bundle freezes the policy, HOG detectors, recognizers and fitted scalers, uncertainty thresholds, feature configuration, environment versions, and artifact hash.

In [3]:
import joblib
payload=joblib.load(BUNDLE/'bundle.joblib'); thresholds={target:{'low':a.low_threshold,'high':a.high_threshold} for target,a in payload['recognizers'].items()}
print(json.dumps(thresholds,indent=2)); print('Bundle:',BUNDLE)


{
  "collared_top": {
    "low": 0.1,
    "high": 0.9000000000000004
  },
  "allowed_sleeve": {
    "low": 0.1,
    "high": 0.9000000000000004
  },
  "casual_round_neck_top": {
    "low": 0.30000000000000004,
    "high": 0.9000000000000004
  },
  "revealing_top": {
    "low": 0.30000000000000004,
    "high": 0.9000000000000004
  },
  "formal_bottom_candidate": {
    "low": 0.1,
    "high": 0.9000000000000004
  },
  "casual_or_tight_bottom": {
    "low": 0.1,
    "high": 0.9000000000000004
  },
  "damaged_bottom": {
    "low": 0.1,
    "high": 0.8500000000000003
  },
  "leisurewear": {
    "low": 0.1,
    "high": 0.8500000000000003
  }
}
Bundle: D:\APU Study\1st Semester\Image Processing and Computer Vision\IPCV Assignment 2\implementation\models\classical-attire-full
